# ProstT5 Baseline Performance — Inverse Folding (3Di→AA) + Folding (AA→3Di)

This notebook establishes the **baseline performance** for **both translation directions** of ProstT5:

1. **Inverse folding (3Di → AA)** — design an AA sequence for a given backbone.
2. **Folding (AA → 3Di)** — predict the 3Di structural states from an AA sequence.

For each direction we benchmark two pipelines:

- **enc–dec**: the full `Rostlab/ProstT5_fp16` encoder–decoder, autoregressive greedy decoding.
- **enc–CNN**: the ProstT5 **encoder only** + a tiny **CNN head** emitting per-residue logits in **one parallel pass** (`model.pt` for 3Di→AA, `model_aa_3di.pt` for AA→3Di).

On a **100-protein test set** (balanced 5×20 over length / organism / function / disease / fold / SS, all ≤1500 residues, built offline by `test_set_100_2.py`), we measure **per direction**:

- **Latency** (s/protein, median over repeats)
- **Throughput** (tokens/s)
- **Peak vRAM** (GB)
- **Greedy agreement** (per-residue identity, confusion matrix, positional analysis)
- **Sampling-based acceptance rate α** (Leviathan's speculative-sampling rule)

Pipeline outline:

1. Mount Google Drive for model caching + checkpointing
2. Load ProstT5 encoder–decoder + both CNN heads
3. Load the pre-built 100-protein test set from FASTA (no AlphaFoldDB / Foldseek run needed)
4. Benchmark latency/throughput/vRAM for both pipelines, both directions
5. Extended agreement evaluation + α measurement across sampling configs

In [ ]:
%pip install tiktoken sentencepiece torch
%pip install 'accelerate>=0.26.0'
%pip install "transformers==4.46.3" "protobuf>=3.20" sentencepiece

In [ ]:
#@title Google Drive mount + HF cache redirect. { display-mode: "form" }
import os

# Prevent transformers from loading TensorFlow (avoids protobuf version conflict on Colab)
os.environ["USE_TF"] = "0"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
    DRIVE_ROOT = "/content/drive/MyDrive/prostT5_benchmarks"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Google Drive mounted. HF cache -> {os.environ['HF_HOME']}")
    print(f"Checkpoint dir -> {DRIVE_ROOT}")
    USE_DRIVE = True
except (ImportError, ModuleNotFoundError):
    print("Not on Colab or Drive unavailable; using local paths.")
    DRIVE_ROOT = None
    USE_DRIVE = False

In [ ]:
#@title Imports + device check. { display-mode: "form" }
import os, time, json, gc, statistics, subprocess, shutil, pickle
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

shutil.rmtree("/kaggle/working/prostT5_benchmarks")
shutil.copytree("/kaggle/input/datasets/vietp253/prostt5-benchmarks", "/kaggle/working/prostT5_benchmarks")


if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch={torch.__version__}  device={device}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

In [ ]:
#@title Configuration. { display-mode: "form" }
PROSTT5_NAME = "Rostlab/ProstT5_fp16"
NOTEBOOK_DIR = Path(".").resolve()

# Locate project files (prefer Drive path on Colab, fallback to local)
if USE_DRIVE:
    PROJECT_DIR = Path(DRIVE_ROOT)
else:
    PROJECT_DIR = NOTEBOOK_DIR


def _find_ckpt(name: str) -> Path:
    """Locate a CNN checkpoint, preferring the project/Drive dir."""
    for candidate in [PROJECT_DIR / name,
                      NOTEBOOK_DIR / name,
                      NOTEBOOK_DIR / "cnn_chkpnt_AA_CNN" / name]:
        if candidate.exists():
            return candidate
    return PROJECT_DIR / name  # default (may not exist yet)


# CNN heads: inverse folding (3Di -> AA) and folding (AA -> 3Di).
CNN_CKPT = _find_ckpt("model.pt")
CNN_FOLD_CKPT = _find_ckpt("model_aa_3di.pt")

DATA_DIR = NOTEBOOK_DIR / "benchmark_data"
RESULTS_DIR = NOTEBOOK_DIR / "benchmark_results"
DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Pre-built test-set FASTAs (generated offline by test_set_100_2.py).
TEST_3DI_FASTA = DATA_DIR / "test_set_3Di_2.fasta"
TEST_AA_FASTA = DATA_DIR / "test_set_AA_2.fasta"

# Checkpoint dir (persistent across Colab restarts if Drive is mounted)
if USE_DRIVE:
    CHECKPOINT_DIR = Path(DRIVE_ROOT) / "checkpoints"
else:
    CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Benchmark protocol
NUM_WARMUP = 0          # per-protein warmup off; a single global warmup runs before the loop
NUM_REPEATS = 3         # timed repeats per protein (we report median)
USE_FP16 = True         # ProstT5 ships an fp16 checkpoint

# Output vocabularies (20 classes each, alphabetical order = ProstT5 CNN class order).
#   inverse folding (3Di->AA): 20 standard amino acids, UPPERCASE
#   folding (AA->3Di):         20 Foldseek 3Di states, lowercase letters a..y
# NOTE: we assume the folding CNN emits its 20 classes in alphabetical 3Di order,
# mirroring the AA CNN. If folding agreement/recovery comes out near zero, the
# class order differs and THREEDI_LETTERS must be reordered to the repo's label order.
AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"
THREEDI_LETTERS = "acdefghiklmnpqrstvwy"
assert len(AA_LETTERS) == 20 and len(THREEDI_LETTERS) == 20

# Sampling configurations for alpha measurement
SAMPLING_CONFIGS = {
    "greedy_T1.0": {"temperature": 1.0, "top_k": 0, "top_p": 1.0},
    "published_T1.0_k3_p0.85": {"temperature": 1.0, "top_k": 3, "top_p": 0.85},
    "conservative_T0.5": {"temperature": 0.5, "top_k": 0, "top_p": 1.0},
    "exploratory_T1.5": {"temperature": 1.5, "top_k": 0, "top_p": 1.0},
}

print(f"AA  CNN checkpoint: {CNN_CKPT}  (exists: {CNN_CKPT.exists()})")
print(f"3Di CNN checkpoint: {CNN_FOLD_CKPT}  (exists: {CNN_FOLD_CKPT.exists()})")
print(f"Project dir: {PROJECT_DIR}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Sampling configs: {list(SAMPLING_CONFIGS.keys())}")

In [ ]:
#@title Load ProstT5 (tokenizer + full encoder–decoder). { display-mode: "form" }
tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)

dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(
    PROSTT5_NAME,
    low_cpu_mem_usage=True,
    torch_dtype=dtype,
).to(device).eval()
if dtype == torch.float16:
    model = model.half()

# We will reuse the same encoder for the enc-CNN path (no second copy of the weights).
encoder = model.get_encoder()

print(f"ProstT5 loaded. dtype={next(model.parameters()).dtype}  "
      f"params(M)={sum(p.numel() for p in model.parameters())/1e6:.1f}")

In [ ]:
#@title Define + load both CNN heads (AA + 3Di) and the Direction configs. { display-mode: "form" }
from dataclasses import dataclass


class CNNHead(nn.Module):
    """Tiny per-residue CNN head over ProstT5 encoder hidden states.
    Conv2d(1024->32, k=7) -> ReLU -> Dropout -> Conv2d(32->20, k=7). Output: 20-class logits."""
    def __init__(self, num_tokens: int = 20, hidden: int = 32, in_dim: int = 1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1).unsqueeze(-1)
        x = self.classifier(x)
        return x.squeeze(-1).permute(0, 2, 1)


def _load_cnn(ckpt_path: Path) -> CNNHead:
    net = CNNHead(num_tokens=20).to(device).eval()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    net.load_state_dict(state_dict, strict=True)
    return net


cnn = _load_cnn(CNN_CKPT)            # inverse folding: 3Di -> AA
cnn_fold = _load_cnn(CNN_FOLD_CKPT)  # folding:         AA  -> 3Di
print(f"AA  CNN loaded ({sum(p.numel() for p in cnn.parameters()):,} params)")
print(f"3Di CNN loaded ({sum(p.numel() for p in cnn_fold.parameters()):,} params)")


@dataclass
class Direction:
    name: str          # "inverse" | "folding"
    prefix: str        # ProstT5 task prefix
    in_key: str        # key in the test_set record holding the INPUT sequence
    ref_key: str       # key holding the reference (ground-truth) OUTPUT sequence
    in_lower: bool     # input residue casing fed to the tokenizer
    out_letters: str   # 20-class output alphabet (CNN class order)
    cnn: nn.Module     # the matching CNN head


DIRECTIONS = {
    "inverse": Direction("inverse", "<fold2AA>", in_key="3di", ref_key="aa",
                         in_lower=True,  out_letters=AA_LETTERS,      cnn=cnn),
    "folding": Direction("folding", "<AA2fold>", in_key="aa",  ref_key="3di",
                         in_lower=False, out_letters=THREEDI_LETTERS, cnn=cnn_fold),
}
print(f"Directions: {list(DIRECTIONS)}")

In [ ]:
#@title Load the test set from pre-built FASTAs. { display-mode: "form" }
# The 100-protein test set is generated offline by `test_set_100_2.py`
# (balanced 5x20 over length/organism/function/disease/fold/SS, all <=1500 residues).
# Here we just load the frozen FASTAs — no AlphaFoldDB download / Foldseek run needed.

def _read_fasta(path: Path) -> dict[str, str]:
    seqs, cur = {}, None
    for line in path.read_text().splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            seqs[cur] = ""
        elif cur is not None:
            seqs[cur] += line.strip()
    return seqs


assert TEST_3DI_FASTA.exists(), f"missing {TEST_3DI_FASTA} (run test_set_100_2.py first)"
assert TEST_AA_FASTA.exists(), f"missing {TEST_AA_FASTA} (run test_set_100_2.py first)"

_aa_seqs = _read_fasta(TEST_AA_FASTA)
_di_seqs = _read_fasta(TEST_3DI_FASTA)

test_set: dict[str, dict] = {}
for uid, aa in _aa_seqs.items():
    di = _di_seqs.get(uid)
    if di is None:
        print(f"  SKIP {uid}: no 3Di entry")
        continue
    aa, di = aa.upper(), di.lower()
    if len(aa) != len(di):
        print(f"  SKIP {uid}: AA/3Di length mismatch ({len(aa)} vs {len(di)})")
        continue
    test_set[uid] = {"aa": aa, "3di": di, "length": len(di)}

lengths = sorted(r["length"] for r in test_set.values())
print(f"Loaded test set: {len(test_set)} proteins")
print(f"Length range: {lengths[0]}-{lengths[-1]} residues  (median {lengths[len(lengths)//2]})")

## Step 2 — Helpers

In [ ]:
#@title Helper: timing utilities + I/O formatting. { display-mode: "form" }
import re


def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def _reset_peak_mem():
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def _peak_mem_gb() -> float:
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated(device) / 1e9
    return 0.0


def _format_input(seq: str, d: "Direction") -> str:
    """ProstT5 prompt: '<prefix> ' + space-separated residues.
    3Di input -> lowercase; AA input -> uppercase with rare residues (U/Z/O/B) mapped to X."""
    if d.in_lower:
        residues = seq.lower()
    else:
        residues = re.sub(r"[UZOB]", "X", seq.upper())
    return f"{d.prefix} " + " ".join(list(residues))


def _decode_seq(token_ids: torch.Tensor, d: "Direction") -> str:
    """Decode generated tokens to a residue string in the direction's output casing."""
    s = "".join(tokenizer.decode(token_ids, skip_special_tokens=True).split())
    return s.upper() if d.out_letters[0].isupper() else s.lower()

In [ ]:
#@title Helper: enc–dec timing (greedy, autoregressive) — both directions. { display-mode: "form" }
@torch.inference_mode()
def time_encdec(uid: str, seq: str, d: "Direction",
                num_warmup: int = NUM_WARMUP,
                num_repeats: int = NUM_REPEATS) -> tuple[list[dict], str]:
    L = len(seq)
    enc = tokenizer(
        [_format_input(seq, d)],
        add_special_tokens=True,
        return_tensors="pt",
    ).to(device)

    gen_kwargs = dict(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        min_length=L + 1,
        num_beams=1,
        do_sample=False,
        num_return_sequences=1,
    )

    for _ in range(num_warmup):
        _ = model.generate(**gen_kwargs)
    _sync()

    rows = []
    last_pred = ""
    for r in range(num_repeats):
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        out = model.generate(**gen_kwargs)
        _sync()
        dt = time.perf_counter() - t0

        n_new = int(out.shape[1] - 1)
        pred = _decode_seq(out[0], d)[:L]  # truncate to expected length
        last_pred = pred
        rows.append({
            "protein_id": uid,
            "direction": d.name,
            "pipeline": "enc_dec",
            "length": L,
            "repeat": r,
            "wall_s": dt,
            "tokens_per_s": n_new / dt if dt > 0 else float("nan"),
            "peak_vram_gb": _peak_mem_gb(),
            "pred_len": len(pred),
        })
    return rows, last_pred

In [ ]:
#@title Helper: enc–CNN timing (encoder + CNN, one parallel pass) — both directions. { display-mode: "form" }
@torch.inference_mode()
def time_enccnn(uid: str, seq: str, d: "Direction",
                num_warmup: int = NUM_WARMUP,
                num_repeats: int = NUM_REPEATS) -> tuple[list[dict], str]:
    L = len(seq)
    enc = tokenizer(
        [_format_input(seq, d)],
        add_special_tokens=True,
        return_tensors="pt",
    ).to(device)

    def _forward() -> torch.Tensor:
        h = encoder(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
        ).last_hidden_state
        h = h[:, 1:-1, :]            # drop the task-prefix token and EOS
        return d.cnn(h.float())

    for _ in range(num_warmup):
        _ = _forward()
    _sync()

    rows = []
    last_pred = ""
    for r in range(num_repeats):
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        logits = _forward()
        ids = logits.argmax(-1)
        _sync()
        dt = time.perf_counter() - t0

        pred = "".join(d.out_letters[int(i)] for i in ids[0].tolist())
        last_pred = pred
        rows.append({
            "protein_id": uid,
            "direction": d.name,
            "pipeline": "enc_cnn",
            "length": L,
            "repeat": r,
            "wall_s": dt,
            "tokens_per_s": len(pred) / dt if dt > 0 else float("nan"),
            "peak_vram_gb": _peak_mem_gb(),
            "pred_len": len(pred),
        })
    return rows, last_pred

In [ ]:
#@title Helper: acceptance rate α computation — both directions. { display-mode: "form" }

# ProstT5 emits space-separated residues, so tokens are ▁A, ▁C, ... (AA) / ▁a, ▁c, ... (3Di).
# convert_tokens_to_ids("A") returns <unk>; must use encode(" A").
OUT_TOKEN_IDS = {
    name: [tokenizer.encode(f" {ch}", add_special_tokens=False)[0] for ch in d.out_letters]
    for name, d in DIRECTIONS.items()
}
for name, ids in OUT_TOKEN_IDS.items():
    print(f"{name}: {DIRECTIONS[name].out_letters[:5]} -> {ids[:5]}")


def _apply_top_k_top_p(logits: torch.Tensor, top_k: int = 0, top_p: float = 1.0) -> torch.Tensor:
    """Apply top-k and top-p (nucleus) filtering to logits. Returns filtered logits."""
    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = float('-inf')
    if top_p < 1.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0
        indices_to_remove = sorted_indices_to_remove.scatter(
            dim=-1, index=sorted_indices, src=sorted_indices_to_remove
        )
        logits[indices_to_remove] = float('-inf')
    return logits


@torch.inference_mode()
def compute_alpha_all_configs(uid: str, seq: str, d: "Direction",
                              sampling_configs: dict) -> dict:
    """α (Leviathan acceptance) of the enc-CNN drafter vs the enc-dec target, for all
    sampling configs, sharing one generate + one teacher-forced pass + one CNN pass.

    The generate and teacher-forced forward are identical across configs (greedy,
    do_sample=False). Only the temperature/top-k/top-p filtering differs.
    """
    L = len(seq)
    out_token_ids = OUT_TOKEN_IDS[d.name]
    enc_input = tokenizer(
        [_format_input(seq, d)], add_special_tokens=True, return_tensors="pt"
    ).to(device)

    # One generate call (greedy) — shared across all configs
    greedy_out = model.generate(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
        max_length=L + 2, do_sample=False, num_beams=1,
    )

    # One teacher-forced forward pass — shared across all configs
    decoder_input_ids = greedy_out[:, :-1]
    enc_out = model(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
        decoder_input_ids=decoder_input_ids,
    )
    p_logits_base = enc_out.logits[0, :L]  # (L, vocab)

    # One CNN forward — shared across all configs
    h = encoder(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
    ).last_hidden_state
    h = h[:, 1:-1, :]
    q_logits_20 = d.cnn(h.float())[0]  # (L_cnn, 20)

    L_eff = min(L, p_logits_base.shape[0], q_logits_20.shape[0])
    results = {}

    for config_name, config_params in sampling_configs.items():
        temperature = config_params.get("temperature", 1.0)
        top_k = config_params.get("top_k", 0)
        top_p = config_params.get("top_p", 1.0)

        p_scaled = p_logits_base[:L_eff].clone() / max(temperature, 1e-8)
        q_scaled = q_logits_20[:L_eff].clone() / max(temperature, 1e-8)

        if top_k > 0 or top_p < 1.0:
            p_scaled = _apply_top_k_top_p(p_scaled, top_k=top_k, top_p=top_p)

        p_probs = torch.softmax(p_scaled, dim=-1)  # (L_eff, vocab)
        q_probs_20 = torch.softmax(q_scaled, dim=-1)  # (L_eff, 20)

        q_probs_full = torch.zeros_like(p_probs)
        for i, tid in enumerate(out_token_ids):
            q_probs_full[:, tid] = q_probs_20[:, i]

        alpha_per_pos = torch.minimum(p_probs, q_probs_full).sum(dim=-1)
        out_mass = p_probs[:, out_token_ids].sum(dim=-1).mean().item()

        results[config_name] = {
            "uid": uid, "direction": d.name, "length": L, "temperature": temperature,
            "top_k": top_k, "top_p": top_p,
            "alpha_mean": alpha_per_pos.mean().item(),
            "alpha_std": alpha_per_pos.std().item(),
            "alpha_min": alpha_per_pos.min().item(),
            "alpha_max": alpha_per_pos.max().item(),
            "p_out_mass_mean": out_mass,
            "L_eff": L_eff,
        }

    return results

In [ ]:
#@title Helper: extended agreement metrics (alphabet-parametrized). { display-mode: "form" }

def confusion_matrix_seq(pred_dec: str, pred_cnn: str, letters: str) -> np.ndarray:
    """KxK confusion matrix: rows = enc-dec (reference), cols = enc-CNN (prediction)."""
    idx = {ch: i for i, ch in enumerate(letters)}
    K = len(letters)
    cm = np.zeros((K, K), dtype=int)
    for dch, cch in zip(pred_dec, pred_cnn):
        if dch in idx and cch in idx:
            cm[idx[dch], idx[cch]] += 1
    return cm


def per_class_metrics(cm: np.ndarray, letters: str) -> dict:
    """Compute precision, recall, F1 per output class from the confusion matrix."""
    metrics = {}
    for i, ch in enumerate(letters):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        metrics[ch] = {"precision": prec, "recall": rec, "f1": f1,
                       "support": int(cm[i, :].sum())}
    return metrics


def positional_agreement(pred_dec: str, pred_cnn: str, n_bins: int = 10) -> list[float]:
    """Agreement as a function of normalized position (0=N-term, 1=C-term)."""
    L = min(len(pred_dec), len(pred_cnn))
    if L == 0:
        return [0.0] * n_bins
    bin_size = L / n_bins
    bin_correct = [0] * n_bins
    bin_total = [0] * n_bins
    for i in range(L):
        b = min(int(i / bin_size), n_bins - 1)
        bin_total[b] += 1
        if pred_dec[i] == pred_cnn[i]:
            bin_correct[b] += 1
    return [c / t if t > 0 else 0.0 for c, t in zip(bin_correct, bin_total)]


print("Extended metrics helpers defined.")

In [ ]:
#@title Checkpointing + progress tracking. { display-mode: "form" }

def save_checkpoint(state: dict, tag: str = "latest"):
    state["timestamp"] = datetime.now().isoformat()
    path = CHECKPOINT_DIR / f"checkpoint_{tag}.pkl"
    with open(path, "wb") as f:
        pickle.dump(state, f)
    latest = CHECKPOINT_DIR / "checkpoint_latest.pkl"
    if tag != "latest":
        with open(latest, "wb") as f:
            pickle.dump(state, f)


def load_checkpoint(tag: str = "latest") -> dict | None:
    path = CHECKPOINT_DIR / f"checkpoint_{tag}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return None


class ProgressTracker:
    def __init__(self, total: int):
        self.total = total
        self.start_time = time.time()
        self.completed = 0
        self.protein_times = []

    def update(self, uid: str, elapsed: float):
        self.completed += 1
        self.protein_times.append(elapsed)
        avg_time = sum(self.protein_times) / len(self.protein_times)
        remaining = (self.total - self.completed) * avg_time
        eta_min = remaining / 60
        total_elapsed = time.time() - self.start_time
        print(f"  [{self.completed}/{self.total}] {uid} took {elapsed:.1f}s | "
              f"ETA: {eta_min:.0f} min | Elapsed: {total_elapsed/60:.0f} min")


print(f"Checkpointing ready. Dir: {CHECKPOINT_DIR}")
ckpt = load_checkpoint()
if ckpt:
    n_done = len(ckpt.get("completed", ckpt.get("completed_proteins", [])))
    print(f"Found existing checkpoint: {n_done} jobs done "
          f"(schema={ckpt.get('schema', 'old')}, from {ckpt.get('timestamp', 'unknown')})")
else:
    print("No existing checkpoint found; starting fresh.")

## Step 3 — Run benchmarks

Resume-aware loop over **both directions**: processes proteins shortest-first, runs inverse + folding per protein, checkpoints after each (protein, direction) job, and skips already-completed work.

In [ ]:
#@title Main benchmark loop (both directions, resume-aware, checkpointed). { display-mode: "form" }

SCHEMA = "v2-bidirectional"  # bump if the checkpoint structure changes

ckpt = load_checkpoint()
if ckpt is not None and ckpt.get("schema") == SCHEMA:
    all_rows = ckpt["all_rows"]
    predictions = ckpt["predictions"]
    alpha_results = ckpt["alpha_results"]
    completed = set(ckpt["completed"])  # keys "uid|direction"
    print(f"Resuming: {len(completed)} (protein, direction) jobs already done.")
else:
    if ckpt is not None:
        print("Checkpoint schema mismatch (old format) -> starting fresh.")
    all_rows, predictions, alpha_results, completed = [], {}, {}, set()

# Shortest-first for quick feedback; both directions per protein.
sorted_proteins = sorted(test_set.items(), key=lambda kv: kv[1]["length"])
jobs = [(uid, rec, dname)
        for uid, rec in sorted_proteins
        for dname in DIRECTIONS
        if f"{uid}|{dname}" not in completed]

print(f"Proteins: {len(sorted_proteins)} x {len(DIRECTIONS)} directions | "
      f"done: {len(completed)} | remaining jobs: {len(jobs)}")

tracker = ProgressTracker(total=len(sorted_proteins) * len(DIRECTIONS))
tracker.completed = len(completed)


# --- Global warmup: compile kernels + ramp GPU clocks once, before any timing ---
# These effects (CUDA kernel compilation, clock ramp, allocator growth) are global,
# not per-protein, so per-protein warmup (NUM_WARMUP) is set to 0 in the config.
@torch.inference_mode()
def _global_warmup(reps: int = 3):
    if device.type == "cpu" or not test_set:
        return
    short = min(test_set.values(), key=lambda r: r["length"])
    for d in DIRECTIONS.values():
        seq = short[d.in_key][:32]  # length-independent: a short slice compiles every kernel
        enc = tokenizer([_format_input(seq, d)], add_special_tokens=True,
                        return_tensors="pt").to(device)
        for _ in range(reps):
            model.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                           max_length=len(seq) + 2, do_sample=False, num_beams=1)
            h = encoder(input_ids=enc.input_ids, attention_mask=enc.attention_mask).last_hidden_state
            d.cnn(h[:, 1:-1, :].float())
    _sync()


_global_warmup()
print("Global warmup done (kernels compiled, clocks ramped).")

for uid, rec, dname in jobs:
    d = DIRECTIONS[dname]
    seq = rec[d.in_key]
    ref = rec[d.ref_key]
    L = rec["length"]
    t_start = time.perf_counter()
    print(f"\n--- [{tracker.completed+1}/{tracker.total}] {uid} [{dname}] L={L} ---")

    rows_dec, pred_dec = time_encdec(uid, seq, d)
    med_dec = statistics.median(r["wall_s"] for r in rows_dec)
    rows_cnn, pred_cnn = time_enccnn(uid, seq, d)
    med_cnn = statistics.median(r["wall_s"] for r in rows_cnn)
    print(f"  enc_dec {med_dec:.2f}s | enc_cnn {med_cnn:.3f}s | speedup {med_dec/med_cnn:.0f}x")

    # Alpha measurement — one generate + one teacher-forced pass shared across all configs
    try:
        alpha_for = compute_alpha_all_configs(uid, seq, d, SAMPLING_CONFIGS)
    except Exception as e:
        print(f"  alpha FAIL: {e}")
        alpha_for = {k: {"alpha_mean": float("nan"), "error": str(e)} for k in SAMPLING_CONFIGS}
    alpha_summary = {k: round(v.get("alpha_mean", float("nan")), 3) for k, v in alpha_for.items()}
    print(f"  alpha: {alpha_summary}")

    # Update state
    all_rows.extend(rows_dec + rows_cnn)
    predictions.setdefault(uid, {})[dname] = {
        "enc_dec": pred_dec,
        "enc_cnn": pred_cnn,
        "reference": ref,
        "length": L,
    }
    alpha_results.setdefault(uid, {})[dname] = alpha_for
    completed.add(f"{uid}|{dname}")

    # Checkpoint
    save_checkpoint({
        "schema": SCHEMA,
        "completed": list(completed),
        "all_rows": all_rows,
        "predictions": predictions,
        "alpha_results": alpha_results,
    })

    tracker.update(f"{uid}[{dname}]", time.perf_counter() - t_start)
    # Keep the caching allocator warm (no empty_cache) so timing stays consistent.
    gc.collect()

print(f"\n{'='*60}")
print(f"BENCHMARK COMPLETE: {len(completed)} jobs processed.")
print(f"Total timed runs: {len(all_rows)}")

In [ ]:
#@title Aggregate + persist results (per direction). { display-mode: "form" }
import pandas as pd

raw = pd.DataFrame(all_rows)
raw.to_csv(RESULTS_DIR / "raw_runs.csv", index=False)

summary = (
    raw.groupby(["direction", "protein_id", "pipeline", "length"])
       .agg(latency_s_median=("wall_s", "median"),
            latency_s_min=("wall_s", "min"),
            tokens_per_s=("tokens_per_s", "median"),
            peak_vram_gb=("peak_vram_gb", "max"))
       .reset_index()
       .sort_values(["direction", "pipeline", "length"])
)
summary.to_csv(RESULTS_DIR / "summary_per_protein.csv", index=False)

# Speedup table (enc-CNN vs enc-dec), per direction
pivot = summary.pivot(index=["direction", "protein_id", "length"],
                      columns="pipeline",
                      values="latency_s_median").reset_index()
pivot["speedup_enc_cnn_over_enc_dec"] = pivot["enc_dec"] / pivot["enc_cnn"]
pivot = pivot.sort_values(["direction", "length"])
pivot.to_csv(RESULTS_DIR / "speedup.csv", index=False)

print("=== Summary statistics ===")
for dname in DIRECTIONS:
    s = summary[summary["direction"] == dname]
    p = pivot[pivot["direction"] == dname]
    if len(p) == 0:
        continue
    print(f"\n[{dname}]  proteins={s['protein_id'].nunique()}")
    print(f"  speedup median/min/max: {p['speedup_enc_cnn_over_enc_dec'].median():.0f}x / "
          f"{p['speedup_enc_cnn_over_enc_dec'].min():.0f}x / "
          f"{p['speedup_enc_cnn_over_enc_dec'].max():.0f}x")
    print(f"  enc_dec throughput (median): {s[s['pipeline']=='enc_dec']['tokens_per_s'].median():.1f} tok/s")
    print(f"  enc_cnn throughput (median): {s[s['pipeline']=='enc_cnn']['tokens_per_s'].median():.1f} tok/s")
    print(f"  enc_dec peak vRAM (max):     {s[s['pipeline']=='enc_dec']['peak_vram_gb'].max():.2f} GB")
    print(f"  enc_cnn peak vRAM (max):     {s[s['pipeline']=='enc_cnn']['peak_vram_gb'].max():.2f} GB")

## Step 4 — Extended Agreement Analysis

In [ ]:
#@title Extended greedy agreement metrics (per direction). { display-mode: "form" }

# Per-protein agreement + a global confusion matrix per direction.
global_cm = {dname: np.zeros((20, 20), dtype=int) for dname in DIRECTIONS}
agree_rows = []

for uid, per_dir in predictions.items():
    for dname, p in per_dir.items():
        letters = DIRECTIONS[dname].out_letters
        L = p["length"]
        a, b = p["enc_dec"][:L], p["enc_cnn"][:L]
        n = min(len(a), len(b))
        if n == 0:
            continue

        # enc-dec ↔ enc-CNN per-residue identity
        identity = sum(1 for i in range(n) if a[i] == b[i]) / n

        # Recovery vs the reference (true AA for inverse, true 3Di for folding)
        ref = p["reference"][:n]
        rec_dec = sum(1 for i in range(len(ref)) if a[i] == ref[i]) / len(ref) if ref else float("nan")
        rec_cnn = sum(1 for i in range(len(ref)) if b[i] == ref[i]) / len(ref) if ref else float("nan")

        global_cm[dname] += confusion_matrix_seq(a[:n], b[:n], letters)

        agree_rows.append({
            "protein_id": uid,
            "direction": dname,
            "length": L,
            "encdec_vs_enccnn_identity": identity,
            "encdec_recovery_vs_ref": rec_dec,
            "enccnn_recovery_vs_ref": rec_cnn,
            "positional_agreement": positional_agreement(a[:n], b[:n]),
        })

agree_df = pd.DataFrame(agree_rows).sort_values(["direction", "length"])
agree_df.drop(columns=["positional_agreement"]).to_csv(RESULTS_DIR / "agreement.csv", index=False)

for dname in DIRECTIONS:
    sub = agree_df[agree_df["direction"] == dname]
    if len(sub) == 0:
        continue
    print(f"\n=== [{dname}] Greedy Agreement ===")
    print(f"  enc-dec ↔ enc-CNN identity:   {sub['encdec_vs_enccnn_identity'].mean():.3f} "
          f"(± {sub['encdec_vs_enccnn_identity'].std():.3f})")
    print(f"  enc-dec recovery vs reference: {sub['encdec_recovery_vs_ref'].mean():.3f}")
    print(f"  enc-CNN recovery vs reference: {sub['enccnn_recovery_vs_ref'].mean():.3f}")
    np.save(RESULTS_DIR / f"confusion_matrix_{dname}.npy", global_cm[dname])

# Per-class metrics (one block per direction)
for dname in DIRECTIONS:
    letters = DIRECTIONS[dname].out_letters
    if global_cm[dname].sum() == 0:
        continue
    m = per_class_metrics(global_cm[dname], letters)
    print(f"\n=== [{dname}] Per-class metrics (enc-CNN predicting enc-dec output) ===")
    print(f"{'cls':>3s}  {'Prec':>6s}  {'Recall':>6s}  {'F1':>6s}  {'Support':>8s}")
    for ch in letters:
        x = m[ch]
        print(f"{ch:>3s}  {x['precision']:6.3f}  {x['recall']:6.3f}  {x['f1']:6.3f}  {x['support']:8d}")

In [ ]:
#@title Alpha (acceptance rate) results summary (per direction). { display-mode: "form" }

alpha_rows = []
for uid, per_dir in alpha_results.items():
    for dname, configs in per_dir.items():
        for config_name, result in configs.items():
            if isinstance(result, dict) and "alpha_mean" in result:
                alpha_rows.append({
                    "protein_id": uid,
                    "direction": dname,
                    "config": config_name,
                    "alpha_mean": result["alpha_mean"],
                    "alpha_std": result.get("alpha_std", float("nan")),
                    "alpha_min": result.get("alpha_min", float("nan")),
                    "alpha_max": result.get("alpha_max", float("nan")),
                    "p_out_mass": result.get("p_out_mass_mean", float("nan")),
                    "length": result.get("length", 0),
                })

alpha_df = pd.DataFrame(alpha_rows)
alpha_df.to_csv(RESULTS_DIR / "alpha_results.csv", index=False)

for dname in DIRECTIONS:
    sub_all = alpha_df[alpha_df["direction"] == dname]
    if len(sub_all) == 0:
        continue
    print(f"\n=== [{dname}] Acceptance Rate α ===")
    print(f"{'Config':<30s}  {'Mean α':>7s}  {'Std':>6s}  {'Min':>6s}  {'Max':>6s}")
    print("-" * 65)
    for config in SAMPLING_CONFIGS.keys():
        s = sub_all[sub_all["config"] == config]
        if len(s) > 0:
            print(f"{config:<30s}  {s['alpha_mean'].mean():7.4f}  {s['alpha_mean'].std():6.4f}  "
                  f"{s['alpha_mean'].min():6.4f}  {s['alpha_mean'].max():6.4f}")
    # Sanity: p(output tokens) should be ~1.0 for a valid alpha computation
    print(f"  sanity: mean p(output tokens) = {sub_all['p_out_mass'].mean():.4f} (should be >0.99)")

    # Predicted speedup from alpha using Theorem 3.8
    print(f"\n  Predicted Speedup from α (Theorem 3.8):")
    print(f"  {'Config':<30s}  {'α':>6s}  {'γ=3':>6s}  {'γ=5':>6s}  {'γ=8':>6s}  {'γ=16':>6s}")
    print("  " + "-" * 73)
    for config in SAMPLING_CONFIGS.keys():
        s = sub_all[sub_all["config"] == config]
        if len(s) == 0:
            continue
        a = s["alpha_mean"].mean()
        gammas = [(1 - a**(g + 1)) / (1 - a) if a < 1.0 else g + 1 for g in [3, 5, 8, 16]]
        print(f"  {config:<30s}  {a:6.3f}  " + "  ".join(f"{x:6.2f}" for x in gammas))

In [ ]:
#@title Plots: latency / throughput / vRAM / agreement / confusion / alpha. { display-mode: "form" }
import matplotlib.pyplot as plt

DIR_MARK = {"inverse": "o", "folding": "s"}


def _scatter_by_pipeline(ax, ycol, ylabel, logy=True):
    for dname in DIRECTIONS:
        for pipeline, sub in summary[summary["direction"] == dname].groupby("pipeline"):
            sub = sub.sort_values("length")
            ax.plot(sub["length"], sub[ycol], DIR_MARK[dname], ms=4, alpha=0.6,
                    label=f"{pipeline} [{dname}]")
    ax.set_xlabel("Protein length (residues)")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log")
    if logy:
        ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=7)


# 1-3) latency / throughput / vRAM (both directions, color by pipeline, marker by direction)
for ycol, ylabel, fname, logy in [
    ("latency_s_median", "Latency (s/protein)", "benchmark_latency.png", True),
    ("tokens_per_s", "Throughput (tokens/s)", "benchmark_throughput.png", True),
    ("peak_vram_gb", "Peak vRAM (GB)", "benchmark_peak_vram.png", False),
]:
    fig, ax = plt.subplots(figsize=(6, 4))
    _scatter_by_pipeline(ax, ycol, ylabel, logy)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / fname, dpi=150)
    plt.show()

# 4) Agreement vs length (both directions)
fig, ax = plt.subplots(figsize=(6, 4))
for dname in DIRECTIONS:
    sub = agree_df[agree_df["direction"] == dname]
    if len(sub) == 0:
        continue
    ax.scatter(sub["length"], sub["encdec_vs_enccnn_identity"], alpha=0.6, s=20,
               marker=DIR_MARK[dname],
               label=f"{dname} (mean={sub['encdec_vs_enccnn_identity'].mean():.3f})")
ax.set_xlabel("Protein length")
ax.set_ylabel("Per-residue identity (enc-dec vs enc-CNN)")
ax.set_title("Greedy agreement vs length")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "benchmark_agreement_vs_length.png", dpi=150)
plt.show()

# 5) Confusion matrix heatmap (one per direction)
for dname in DIRECTIONS:
    if global_cm[dname].sum() == 0:
        continue
    letters = DIRECTIONS[dname].out_letters
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(global_cm[dname], cmap="Blues", aspect="auto")
    ax.set_xticks(range(20)); ax.set_yticks(range(20))
    ax.set_xticklabels(list(letters), fontsize=7)
    ax.set_yticklabels(list(letters), fontsize=7)
    ax.set_xlabel("enc-CNN prediction")
    ax.set_ylabel("enc-dec output")
    ax.set_title(f"Confusion matrix [{dname}] (20 classes)")
    plt.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"benchmark_confusion_matrix_{dname}.png", dpi=150)
    plt.show()

# 6) Alpha vs length (greedy_T1.0, both directions)
fig, ax = plt.subplots(figsize=(6, 4))
if len(alpha_df) > 0:
    for dname in DIRECTIONS:
        s = alpha_df[(alpha_df["direction"] == dname) & (alpha_df["config"] == "greedy_T1.0")].sort_values("length")
        if len(s) > 0:
            ax.scatter(s["length"], s["alpha_mean"], alpha=0.6, s=18,
                       marker=DIR_MARK[dname], label=f"{dname} (greedy_T1.0)")
    ax.set_xlabel("Protein length")
    ax.set_ylabel("α (acceptance rate)")
    ax.set_title("α vs length (greedy_T1.0)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "benchmark_alpha_vs_length.png", dpi=150)
plt.show()

print(f"Saved plots to {RESULTS_DIR}")

In [ ]:
#@title Positional agreement analysis (per direction). { display-mode: "form" }

fig, ax = plt.subplots(figsize=(8, 4))
plotted = False
for dname in DIRECTIONS:
    rows = [r["positional_agreement"]
            for _, r in agree_df[agree_df["direction"] == dname].iterrows()
            if isinstance(r.get("positional_agreement"), list)]
    if not rows:
        continue
    n_bins = len(rows[0])
    mean_pos = [np.mean([p[i] for p in rows]) for i in range(n_bins)]
    std_pos = [np.std([p[i] for p in rows]) for i in range(n_bins)]
    x = np.linspace(0, 1, n_bins)
    ax.errorbar(x, mean_pos, yerr=std_pos, fmt=DIR_MARK[dname] + "-", capsize=3,
                alpha=0.8, label=dname)
    plotted = True

if plotted:
    ax.set_xlabel("Normalized position (0=N-term, 1=C-term)")
    ax.set_ylabel("Agreement (enc-dec vs enc-CNN)")
    ax.set_title("Positional agreement profile (averaged over proteins)")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "positional_agreement.png", dpi=150)
    plt.show()
    print(f"Saved plot to {RESULTS_DIR / 'positional_agreement.png'}")
else:
    print("No positional data available.")